**⭐ 1. What This Pattern Solves**

Splits raw log lines into structured pieces for ETL or analytics pipelines.
Enables field extraction for metrics, monitoring, or downstream transformations.
Handles heterogeneous log formats (e.g., CSV, JSON, Apache/Nginx, custom).
Prepares data for aggregations, joins, or filtering in batch/stream pipelines.

**⭐ 2. SQL Equivalent**

In [0]:
%sql
-- Extract IP and timestamp from log string using SUBSTRING / SPLIT_PART
SELECT
    SPLIT_PART(log_line, ' ', 1) AS ip,
    SPLIT_PART(log_line, ' ', 4) AS timestamp
FROM logs;

**⭐ 3. Core Idea**

Treat logs as strings → split on delimiters or regex → map to fields.
Works because structured patterns can be consistently extracted from text.

**⭐ 4. Template Code (MEMORIZE THIS)**

In [0]:
import re

def tokenize_logs(logs, pattern):
    """
    logs: iterable of log lines
    pattern: regex pattern with capture groups for fields
    """
    for line in logs:
        match = re.match(pattern, line)
        if match:
            yield match.groups()

**⭐ 5. Detailed Example**

In [0]:
logs = [
    '127.0.0.1 - - [16/Dec/2025:12:00:01] "GET /home HTTP/1.1" 200 512',
    '192.168.1.5 - - [16/Dec/2025:12:00:02] "POST /login HTTP/1.1" 302 128'
]

pattern = r'(\S+) - - \[(.*?)\] "(.*?)" (\d+) (\d+)'

for ip, timestamp, request, status, size in tokenize_logs(logs, pattern):
    print(ip, timestamp, request, status, size)

127.0.0.1 16/Dec/2025:12:00:01 GET /home HTTP/1.1 200 512
192.168.1.5 16/Dec/2025:12:00:02 POST /login HTTP/1.1 302 128

**⭐ 6. Mini Practice Problems**

Tokenize CSV-formatted logs: "2025-12-16,ERROR,User login failed" → (date, level, message)
Extract IPs and URLs from web server logs.
Parse JSON log lines into (timestamp, user_id, action) tuples.

**⭐ 7. Full Data Engineering Scenario**

Problem: Web server logs are written to S3 in raw text. Analytics team needs daily counts of HTTP status codes per URL.

Expected Output:

In [0]:
/home 200 150
/login 302 30
/dashboard 404 5

pattern = r'(\S+) - - \[(.*?)\] "(GET|POST) (\S+) HTTP/1.1" (\d+) (\d+)'

status_counts = defaultdict(int)
for ip, timestamp, method, url, status, size in tokenize_logs(s3_logs, pattern):
    status_counts[(url, status)] += 1

**⭐ 8. Time & Space Complexity**

Time Complexity: O(n * m) → n = number of log lines, m = average line length (regex match scans string).
Space Complexity: O(n) if storing all results; O(1) if streaming/yielding.

**⭐ 9. Common Pitfalls & Mistakes**

❌ Using split(' ') blindly → fails on logs with spaces inside quotes.
❌ Storing all parsed logs in memory unnecessarily.
❌ Forgetting to handle missing or malformed lines → pipeline crashes.
✔ Use regex with capture groups.
✔ Yield results for streaming ETL.
✔ Add error handling for bad lines.